In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
for candidate in [project_root, project_root.parent]:
    if (candidate / "app").exists():
        project_root = candidate
        break
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.core.db import db_connection


def _execute_sql(sql: str, params: tuple | None = None) -> list[tuple]:
    with db_connection() as connection:
        cursor = connection.cursor()
        cursor.execute(sql, params or ())
        rows = cursor.fetchall()
        cursor.close()
        return rows


def _load_dates() -> pd.DataFrame:
    rows = _execute_sql("SELECT fecha, año AS ano, semana FROM dates WHERE dia_semana=1")
    df_dates = pd.DataFrame(rows, columns=["fecha", "ano", "semana"])
    if df_dates.empty:
        return df_dates
    df_dates["ano"] = pd.to_numeric(df_dates["ano"], errors="coerce").astype("Int64")
    df_dates["semana"] = pd.to_numeric(df_dates["semana"], errors="coerce").astype("Int64")
    return df_dates


def _load_varieties() -> pd.DataFrame:
    rows = _execute_sql("SELECT variedad, ciclo, curva, s1, s2, s3 FROM curva_variedad_rosas WHERE activo=1")
    df_varieties = pd.DataFrame(
        rows,
        columns=["variedad", "ciclo", "curva_distribucion", "S1", "S2", "S3"],
    )
    if df_varieties.empty:
        return df_varieties
    for column in ["curva_distribucion", "S1", "S2", "S3"]:
        df_varieties[column] = df_varieties[column].apply(
            lambda value: json.loads(value) if isinstance(value, str) and value.strip() else []
        )
    return df_varieties


def proyectar_tallos(df_historico: pd.DataFrame, df_varieties: pd.DataFrame, projection_kind: str, fecha_inicio, fecha_fin) -> pd.DataFrame:
    if df_historico.empty:
        return pd.DataFrame(columns=["variedad", "bloque", "tipo", "fecha_proyeccion", "cantidad_proyectada"])

    df = df_historico.copy()
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce").dt.normalize()
    df = df[df["fecha"].notna()].copy()
    df = df[(df["fecha"] >= pd.to_datetime(fecha_inicio)) & (df["fecha"] <= pd.to_datetime(fecha_fin))].copy()
    if df.empty:
        return pd.DataFrame(columns=["variedad", "bloque", "tipo", "fecha_proyeccion", "cantidad_proyectada"])

    grouped = df.groupby(["variedad", "bloque"], as_index=False)["tallos"].mean().rename(columns={"tallos": "cantidad_promedio"})
    if not df_varieties.empty:
        grouped = grouped.merge(df_varieties[["variedad", "curva_distribucion"]], on="variedad", how="left")

    projection_dates = pd.date_range(start=pd.to_datetime(fecha_inicio).normalize(), end=pd.to_datetime(fecha_fin).normalize(), freq="W-MON")
    if projection_dates.empty:
        projection_dates = pd.DatetimeIndex([pd.to_datetime(fecha_fin).normalize()])

    rows: list[dict] = []
    for _, row in grouped.iterrows():
        variedad = row["variedad"]
        bloque = row["bloque"]
        promedio = float(row.get("cantidad_promedio", 0.0) or 0.0)
        curve = row.get("curva_distribucion")
        if isinstance(curve, str):
            try:
                curve = json.loads(curve)
            except Exception:
                curve = []
        if not isinstance(curve, list) or not curve:
            curve = [1.0]
        weights = pd.Series(curve, dtype=float)
        if weights.sum() == 0:
            weights = pd.Series([1.0] * len(weights) if len(weights) else [1.0], dtype=float)
        weights = weights / weights.sum()

        for i, projection_date in enumerate(projection_dates):
            weight = float(weights.iloc[i % len(weights)]) if len(weights) else 1.0
            rows.append(
                {
                    "variedad": variedad,
                    "bloque": bloque,
                    "tipo": projection_kind,
                    "fecha_proyeccion": pd.Timestamp(projection_date).normalize(),
                    "cantidad_proyectada": promedio * weight,
                }
            )

    return pd.DataFrame(rows, columns=["variedad", "bloque", "tipo", "fecha_proyeccion", "cantidad_proyectada"])


In [5]:
import json
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
for candidate in [project_root, project_root.parent]:
    if (candidate / "app").exists():
        project_root = candidate
        break
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.core.db import db_connection


def _execute_sql(sql: str, params: tuple | None = None) -> list[tuple]:
    with db_connection() as connection:
        cursor = connection.cursor()
        cursor.execute(sql, params or ())
        rows = cursor.fetchall()
        cursor.close()
        return rows


def _load_dates() -> pd.DataFrame:
    rows = _execute_sql("SELECT fecha, año AS ano, semana FROM dates WHERE dia_semana=1")
    df_dates = pd.DataFrame(rows, columns=["fecha", "ano", "semana"])
    if df_dates.empty:
        return df_dates
    df_dates["ano"] = pd.to_numeric(df_dates["ano"], errors="coerce").astype("Int64")
    df_dates["semana"] = pd.to_numeric(df_dates["semana"], errors="coerce").astype("Int64")
    return df_dates


def _load_varieties() -> pd.DataFrame:
    rows = _execute_sql("SELECT variedad, ciclo, curva, s1, s2, s3 FROM curva_variedad_rosas WHERE activo=1")
    df_varieties = pd.DataFrame(
        rows,
        columns=["variedad", "ciclo", "curva_distribucion", "S1", "S2", "S3"],
    )
    if df_varieties.empty:
        return df_varieties
    for column in ["curva_distribucion", "S1", "S2", "S3"]:
        df_varieties[column] = df_varieties[column].apply(
            lambda value: json.loads(value) if isinstance(value, str) and value.strip() else []
        )
    return df_varieties


def proyectar_tallos(df_historico: pd.DataFrame, df_varieties: pd.DataFrame, projection_kind: str, fecha_inicio, fecha_fin) -> pd.DataFrame:
    if df_historico.empty:
        return pd.DataFrame(columns=["variedad", "bloque", "tipo", "fecha_proyeccion", "cantidad_proyectada"])

    df = df_historico.copy()
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce").dt.normalize()
    df = df[df["fecha"].notna()].copy()
    df = df[(df["fecha"] >= pd.to_datetime(fecha_inicio)) & (df["fecha"] <= pd.to_datetime(fecha_fin))].copy()
    if df.empty:
        return pd.DataFrame(columns=["variedad", "bloque", "tipo", "fecha_proyeccion", "cantidad_proyectada"])

    grouped = df.groupby(["variedad", "bloque"], as_index=False)["tallos"].mean().rename(columns={"tallos": "cantidad_promedio"})
    if not df_varieties.empty:
        grouped = grouped.merge(df_varieties[["variedad", "curva_distribucion"]], on="variedad", how="left")

    projection_dates = pd.date_range(start=pd.to_datetime(fecha_inicio).normalize(), end=pd.to_datetime(fecha_fin).normalize(), freq="W-MON")
    if projection_dates.empty:
        projection_dates = pd.DatetimeIndex([pd.to_datetime(fecha_fin).normalize()])

    rows: list[dict] = []
    for _, row in grouped.iterrows():
        variedad = row["variedad"]
        bloque = row["bloque"]
        promedio = float(row.get("cantidad_promedio", 0.0) or 0.0)
        curve = row.get("curva_distribucion")
        if isinstance(curve, str):
            try:
                curve = json.loads(curve)
            except Exception:
                curve = []
        if not isinstance(curve, list) or not curve:
            curve = [1.0]
        weights = pd.Series(curve, dtype=float)
        if weights.sum() == 0:
            weights = pd.Series([1.0] * len(weights) if len(weights) else [1.0], dtype=float)
        weights = weights / weights.sum()

        for i, projection_date in enumerate(projection_dates):
            weight = float(weights.iloc[i % len(weights)]) if len(weights) else 1.0
            rows.append(
                {
                    "variedad": variedad,
                    "bloque": bloque,
                    "tipo": projection_kind,
                    "fecha_proyeccion": pd.Timestamp(projection_date).normalize(),
                    "cantidad_proyectada": promedio * weight,
                }
            )

    return pd.DataFrame(rows, columns=["variedad", "bloque", "tipo", "fecha_proyeccion", "cantidad_proyectada"])


df_dates = _load_dates()

datos = pd.DataFrame(
    _execute_sql("SELECT fecha, variedad, bloque, tallos, tipo FROM ld_proyecciones WHERE flor IN ('ROC','ROS')"),
    columns=["fecha", "variedad", "bloque", "tallos", "tipo"],
)
if not datos.empty:
    datos["fecha"] = pd.to_datetime(datos["fecha"], errors="coerce").dt.normalize()
    datos["bloque"] = pd.to_numeric(datos["bloque"], errors="coerce")
    datos["tallos"] = pd.to_numeric(datos["tallos"], errors="coerce")


In [6]:
df_cuts = (
    datos[datos["tipo"].isin(["RE", "AJ"]) ]
    .groupby(["fecha", "variedad", "bloque", "tipo"], as_index=False)["tallos"]
    .sum()
)
if not df_cuts.empty:
    df_cuts = df_cuts.merge(df_dates, on="fecha", how="left")

# Se separan reales y proyecciones, pero la comparación final no usa AJ.
df_cortes_reales = df_cuts[df_cuts["tipo"] == "RE"].copy() if not df_cuts.empty else pd.DataFrame(columns=["fecha", "variedad", "bloque", "tipo", "tallos", "ano", "semana"])
df_projection = df_cuts[df_cuts["tipo"] == "AJ"].copy() if not df_cuts.empty else pd.DataFrame(columns=["fecha", "variedad", "bloque", "tipo", "tallos", "ano", "semana"])

In [7]:
df_pruning = pd.DataFrame(
    _execute_sql("SELECT fecha, nombre_variedad AS variedad, bloque, tallos, tipo FROM ld_arre WHERE flor IN ('ROC','ROS')"),
    columns=["fecha", "variedad", "bloque", "tallos", "tipo"],
)
if not df_pruning.empty:
    df_pruning["fecha"] = pd.to_datetime(df_pruning["fecha"], errors="coerce").dt.normalize()
    df_pruning["bloque"] = pd.to_numeric(df_pruning["bloque"], errors="coerce")
    df_pruning["tallos"] = pd.to_numeric(df_pruning["tallos"], errors="coerce")
    df_pruning = df_pruning.merge(df_dates, on="fecha", how="left")

# Export auxiliar no depende de un Excel externo; toma los datos desde ld_arre y ld_proyecciones.
df_export_arreglos_corte = pd.concat(
    [
        df_pruning[["variedad", "bloque", "tipo", "ano", "semana", "tallos"]],
        df_cuts[["variedad", "bloque", "tipo", "ano", "semana", "tallos"]],
    ],
    axis=0,
    ignore_index=True,
)

output_dir = Path("C:/Users/ESTADISTICA/Downloads")
output_dir.mkdir(parents=True, exist_ok=True)
df_export_arreglos_corte.to_excel(output_dir / "arreglos y cortes.xlsx", index=False)

In [8]:
df_varieties = _load_varieties()

In [9]:
df_podas_reales = df_pruning[["fecha", "bloque", "variedad", "tipo", "tallos"]].groupby(["fecha", "bloque", "variedad", "tipo"], as_index=False)["tallos"].sum()
if not df_podas_reales.empty:
    df_podas_reales["fecha"] = pd.to_datetime(df_podas_reales["fecha"], errors="coerce").dt.normalize()

# Selectivas: S1, S2, S3
if not df_podas_reales.empty:
    df_podas_selectivas = df_podas_reales[df_podas_reales["tipo"].isin(["S1", "S2", "S3"])].copy()
else:
    df_podas_selectivas = pd.DataFrame(columns=["fecha", "bloque", "variedad", "tipo", "tallos"])

In [10]:
fecha_inicio = pd.to_datetime("2024-01-01")
fecha_fin = pd.to_datetime("2026-12-31")

df_podas_reales["fecha"] = pd.to_datetime(df_podas_reales["fecha"], errors="coerce")
df_cortes_reales["fecha"] = pd.to_datetime(df_cortes_reales["fecha"], errors="coerce")
df_podas_selectivas["fecha"] = pd.to_datetime(df_podas_selectivas["fecha"], errors="coerce")

proyeccion_podas = proyectar_tallos(df_podas_reales, df_varieties, "Poda", fecha_inicio, fecha_fin)
proyeccion_cortes = proyectar_tallos(df_cortes_reales, df_varieties, "Corte", fecha_inicio, fecha_fin)
proyeccion_selectiva = proyectar_tallos(df_podas_selectivas, df_varieties, "Selectiva", fecha_inicio, fecha_fin)
proyeccion_total = pd.concat([proyeccion_podas, proyeccion_cortes, proyeccion_selectiva], ignore_index=True)

In [11]:
proyeccion_total = proyeccion_total.groupby(["variedad", "bloque", "fecha_proyeccion"], as_index=False)["cantidad_proyectada"].sum()

exportar = pd.merge(
    proyeccion_total,
    df_cortes_reales[["fecha", "bloque", "variedad", "tallos"]].rename(columns={"fecha": "fecha_proyeccion"}),
    on=["fecha_proyeccion", "bloque", "variedad"],
    how="outer",
).rename(columns={"tallos": "tallos_real"})

# AJ se deja fuera por decisión del negocio; no se usa para comparar ni para exportar.
exportar = exportar.rename(columns={"cantidad_proyectada": "tallos_formula"})

In [12]:
exportar.head()

,variedad,bloque,fecha_proyeccion,tallos_formula,tallos_real
0,FREEDOM,1.0,2024-01-01,1463.295109,18502.0
1,PECKOUBO,1.0,2024-01-01,271.775399,6049.0
2,MOMENTUM,2.0,2024-01-01,289.906238,2870.0
3,MONDIAL,2.0,2024-01-01,110.735188,NaN
4,VENDELA,2.0,2024-01-01,846.064096,20097.0


In [13]:
exportar.to_excel("C:/Users/ESTADISTICA/Downloads/proyeccion rosas.xlsx", index=False)

## Proyección de arreglos

En este notebook se usa la base de datos directamente y no se depende de Excel ni de archivos intermedios para construir los arreglos o la proyección.